# Kaggle Full Notebook - Embed `corpus_important_docs_chunks.jsonl`

This notebook runs end-to-end embedding on Kaggle and exports:
- `embeddings_*.npy`
- `metadata_*.parquet` (or jsonl/csv fallback)
- `manifest.json`
- optional `emb_out.zip`


In [ ]:
!nvidia-smi
!pip install -q sentence-transformers pyarrow pandas tqdm

In [ ]:
from pathlib import Path
import json
import time
from typing import Any

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

def find_input_jsonl(explicit_path: str | None = None) -> Path:
    if explicit_path:
        p = Path(explicit_path)
        if p.exists():
            return p

    root = Path('/kaggle/input')
    if not root.exists():
        raise FileNotFoundError('/kaggle/input does not exist')

    candidates = list(root.rglob('corpus_important_docs_chunks.jsonl'))
    if not candidates:
        raise FileNotFoundError(
            'Cannot find corpus_important_docs_chunks.jsonl under /kaggle/input. '
            'Set INPUT_JSONL manually in the next cell.'
        )

    candidates.sort(key=lambda p: len(str(p)))
    return candidates[0]


In [ ]:
# ===== Config =====
INPUT_JSONL = find_input_jsonl()
# INPUT_JSONL = Path('/kaggle/input/<your-dataset>/corpus_important_docs_chunks.jsonl')

OUTPUT_DIR = Path('/kaggle/working/emb_out')

MODEL_NAME = 'intfloat/multilingual-e5-base'
DEVICE = 'cuda'
BATCH_SIZE = 96      # set 0 for auto
MAX_LENGTH = 384
NORMALIZE = True
PREFIX_MODE = 'auto' # auto | query_passage | none

EMB_DTYPE = 'float16'      # float16 | float32
METADATA_FORMAT = 'parquet' # parquet | jsonl | csv
SHARD_SIZE = 50000
LIMIT = 0                  # 0 = full
LOG_EVERY = 10

TEXT_KEY = 'chunk_text'
ID_KEY = 'chunk_id'
META_KEYS = [
    'chunk_id','id','document_number','title','url','legal_type','legal_sectors',
    'issuing_authority','issuance_date','signers','section_type','article_no','clause_no',
    'word_count','token_estimate'
]

print('INPUT_JSONL:', INPUT_JSONL)
print('INPUT exists:', INPUT_JSONL.exists())
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
def infer_batch_size(model_name: str, user_batch_size: int) -> int:
    if user_batch_size > 0:
        return user_batch_size
    m = model_name.lower()
    if 'e5-small' in m:
        return 128
    if 'e5-base' in m:
        return 96
    return 64

def infer_prefix_mode(model_name: str, mode: str) -> str:
    if mode != 'auto':
        return mode
    return 'query_passage' if 'e5' in model_name.lower() else 'none'

def count_jsonl(path: Path) -> int:
    c = 0
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                c += 1
    return c

def iter_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def write_metadata(rows: list[dict[str, Any]], out_stub: Path, fmt: str) -> str:
    out_stub.parent.mkdir(parents=True, exist_ok=True)
    if fmt == 'jsonl':
        out = out_stub.with_suffix('.jsonl')
        with out.open('w', encoding='utf-8') as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + '\n')
        return out.name

    if fmt == 'csv':
        out = out_stub.with_suffix('.csv')
        pd.DataFrame(rows).to_csv(out, index=False)
        return out.name

    out = out_stub.with_suffix('.parquet')
    try:
        pd.DataFrame(rows).to_parquet(out, index=False)
        return out.name
    except Exception:
        out = out_stub.with_suffix('.jsonl')
        with out.open('w', encoding='utf-8') as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + '\n')
        return out.name

def flush_shard(shard_id: int, emb_parts: list[np.ndarray], meta_rows: list[dict[str, Any]], output_dir: Path, emb_dtype: np.dtype, metadata_format: str):
    arr = np.concatenate(emb_parts, axis=0)
    if arr.dtype != emb_dtype:
        arr = arr.astype(emb_dtype, copy=False)

    emb_name = f'embeddings_{shard_id:05d}.npy'
    np.save(output_dir / emb_name, arr)

    meta_name = write_metadata(meta_rows, output_dir / f'metadata_{shard_id:05d}', metadata_format)

    return {
        'shard_id': shard_id,
        'rows': int(arr.shape[0]),
        'dim': int(arr.shape[1]),
        'embedding_file': emb_name,
        'metadata_file': meta_name,
        'embedding_dtype': str(arr.dtype),
    }

In [ ]:
def run_embedding():
    assert INPUT_JSONL.exists(), f'Input not found: {INPUT_JSONL}'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    batch_size = infer_batch_size(MODEL_NAME, BATCH_SIZE)
    prefix_mode = infer_prefix_mode(MODEL_NAME, PREFIX_MODE)
    emb_dtype = np.float16 if EMB_DTYPE == 'float16' else np.float32

    print('Loading model:', MODEL_NAME)
    model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    if hasattr(model, 'max_seq_length'):
        model.max_seq_length = MAX_LENGTH

    total_rows = count_jsonl(INPUT_JSONL)
    if LIMIT > 0:
        total_rows = min(total_rows, LIMIT)

    print('Rows to process:', total_rows)
    print(f'Device={DEVICE} | batch_size={batch_size} | max_length={MAX_LENGTH}')
    print(f'prefix_mode={prefix_mode} | normalize={NORMALIZE} | emb_dtype={EMB_DTYPE}')

    emb_parts = []
    meta_rows = []
    shards = []

    batch_texts = []
    batch_meta = []

    processed = 0
    shard_id = 0
    started = time.time()

    def process_batch():
        nonlocal batch_texts, batch_meta
        if not batch_texts:
            return
        embs = model.encode(
            batch_texts,
            batch_size=batch_size,
            show_progress_bar=False,
            normalize_embeddings=NORMALIZE,
            convert_to_numpy=True,
        )
        emb_parts.append(embs)
        meta_rows.extend(batch_meta)
        batch_texts = []
        batch_meta = []

    pbar = tqdm(total=total_rows, desc='Embedding', unit='rows')

    for row in iter_jsonl(INPUT_JSONL):
        if LIMIT > 0 and processed >= LIMIT:
            break

        raw_text = str(row.get(TEXT_KEY, '')).strip()
        if not raw_text:
            continue

        text = f'passage: {raw_text}' if prefix_mode == 'query_passage' else raw_text
        batch_texts.append(text)
        batch_meta.append({k: row.get(k) for k in META_KEYS})

        if len(batch_texts) >= batch_size:
            process_batch()

        processed += 1
        pbar.update(1)

        current_rows = sum(x.shape[0] for x in emb_parts)
        if current_rows >= SHARD_SIZE:
            info = flush_shard(shard_id, emb_parts, meta_rows, OUTPUT_DIR, emb_dtype, METADATA_FORMAT)
            shards.append(info)
            shard_id += 1
            emb_parts = []
            meta_rows = []

            if LOG_EVERY > 0 and shard_id % LOG_EVERY == 0:
                elapsed = time.time() - started
                speed = processed / max(elapsed, 1e-6)
                print(f'Shards={shard_id} | processed={processed} | speed={speed:.1f} rows/s')

    process_batch()

    if emb_parts:
        info = flush_shard(shard_id, emb_parts, meta_rows, OUTPUT_DIR, emb_dtype, METADATA_FORMAT)
        shards.append(info)

    pbar.close()

    elapsed = time.time() - started
    manifest = {
        'input_jsonl': str(INPUT_JSONL),
        'model_name': MODEL_NAME,
        'device': DEVICE,
        'max_length': MAX_LENGTH,
        'prefix_mode': prefix_mode,
        'normalize': NORMALIZE,
        'embedding_dtype': EMB_DTYPE,
        'metadata_format': METADATA_FORMAT,
        'text_key': TEXT_KEY,
        'id_key': ID_KEY,
        'meta_keys': META_KEYS,
        'total_rows': int(sum(s['rows'] for s in shards)),
        'embedding_dim': int(shards[0]['dim']) if shards else 0,
        'shard_size_target': SHARD_SIZE,
        'shards': shards,
        'elapsed_sec': elapsed,
        'created_at_unix': int(time.time()),
    }

    manifest_path = OUTPUT_DIR / 'manifest.json'
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

    print('Done')
    print('total_rows:', manifest['total_rows'])
    print('embedding_dim:', manifest['embedding_dim'])
    print('num_shards:', len(shards))
    print('elapsed_sec:', round(elapsed, 1))
    print('output_dir:', OUTPUT_DIR)
    print('manifest:', manifest_path)

    return manifest

manifest = run_embedding()

In [ ]:
!ls -lah {OUTPUT_DIR}

m = json.loads((OUTPUT_DIR / 'manifest.json').read_text(encoding='utf-8'))
print('manifest total_rows:', m['total_rows'])
print('manifest embedding_dim:', m['embedding_dim'])
print('manifest shards:', len(m['shards']))

In [ ]:
# Optional: zip output for download from Kaggle UI
!cd /kaggle/working && zip -r emb_out.zip emb_out -q
!ls -lh /kaggle/working/emb_out.zip